<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/systems/int8-quantization/int8-quantization.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("int8-quantization", ...)`


# INT8 Quantization from Scratch

**Difficulty**: 🟡 Medium

### Problem Statement

Post-training quantization (PTQ) is how most large models are served: int8 weights halve
memory and HBM traffic relative to bf16 (int4 quarters it), and int8 tensor cores run about
twice the bf16 FLOP rate. PyTorch's `torch.ao.quantization`, TensorRT, LLM.int8(), SmoothQuant,
GPTQ and AWQ all build on the same **affine map** between a real value $x$ and an integer code $q$:

$$\hat x = s\,(q - z), \qquad q = \operatorname{clamp}\!\big(\operatorname{round}(x / s) + z,\; q_{\min},\; q_{\max}\big)$$

with a positive **scale** $s$ and an integer **zero point** $z$. Rounding to the nearest code
bounds the error by half a step: $|\hat x - x| \le s/2$ for every $x$ inside the representable
range.

The payoff is that a linear layer runs on integers. With activations $\hat x = s_x (q^x - z_x)$
(per-tensor) and weights $\hat W_{j\cdot} = s_{w,j}\, q^w_{j\cdot}$ (symmetric, one scale per output
channel $j$):

$$y_{ij} = \sum_k \hat x_{ik}\,\hat W_{jk} = s_x\, s_{w,j} \underbrace{\sum_k \big(q^x_{ik} - z_x\big)\, q^w_{jk}}_{\text{exact integer accumulator (int32)}}$$

— all the multiply-adds happen in integer arithmetic and the only floating-point work is a
single rescale per output.

---

### Requirements

1. **`quantize_symmetric(x, num_bits=8, axis=None) -> (q, scale)`**
   - $q_{\max} = 2^{b-1} - 1$ (127 for 8 bits). Codes live in $[-q_{\max}, q_{\max}]$ — the code
     $-2^{b-1}$ is left unused so the grid is symmetric — and the zero point is 0.
   - `axis=None` (per-tensor): $s = \max|x| \,/\, q_{\max}$, returned as a 0-dim float tensor.
   - `axis=k` (per-channel, `k` may be negative): one scale per index of dimension `k`, computed
     from the max of $|x|$ over **all other** dimensions; returned as a 1-D tensor of shape
     `(x.shape[k],)`. For an `nn.Linear` weight `(out_features, in_features)`, `axis=0` is
     per-output-channel.
   - $q = \operatorname{clamp}(\operatorname{round}(x / s), -q_{\max}, q_{\max})$, stored as
     `torch.int8` (also for `num_bits < 8`, e.g. 4-bit codes in $[-7, 7]$).
   - An all-zero tensor or channel must still get a positive, finite scale (use 1.0) — never NaN.
2. **`quantize_asymmetric(x, num_bits=8) -> (q, scale, zero_point)`** (per-tensor)
   - Codes in $[0, 2^b - 1]$, stored as `torch.uint8`.
   - Widen the range to contain zero: $x_{\min} = \min(\min x, 0)$, $x_{\max} = \max(\max x, 0)$,
     and $s = (x_{\max} - x_{\min}) / (2^b - 1)$ (1.0 if $x_{\max} = x_{\min}$).
   - `zero_point` is an **integer** (Python `int` or 0-dim integer tensor):
     $z = \operatorname{clamp}(\operatorname{round}(-x_{\min}/s), 0, 2^b - 1)$, so that $0.0$ is
     exactly representable (it quantizes to $z$ and dequantizes to exactly 0).
   - $q = \operatorname{clamp}(\operatorname{round}(x / s) + z,\; 0,\; 2^b - 1)$.
3. **`dequantize(q, scale, zero_point=None, axis=None) -> Tensor`** — float32 $(q - z)\,s$;
   `zero_point=None` means 0. With `axis`, `scale` is 1-D and broadcasts along that dimension.
4. **`int8_linear(x_q, x_scale, x_zero_point, w_q, w_scale) -> Tensor`**
   - `x_q`: `(M, K)` uint8 activation codes, quantized per-tensor with `x_scale` and
     `x_zero_point` (`None` means 0). `w_q`: `(N, K)` int8 weight codes (the `nn.Linear`
     layout), symmetric, with `w_scale` a scalar or a `(N,)` tensor (per output channel).
   - Return the float32 `(M, N)` result $\approx \hat x\, \hat W^\top$: compute
     $\sum_k (q^x_{ik} - z_x)\, q^w_{jk}$ **exactly with integer arithmetic**, then multiply once by
     `x_scale * w_scale` (a per-channel `w_scale` scales the output columns).
   - "Exactly" is tested: with `x_scale = w_scale = 1` the output must equal the exact integer
     dot products (rounded once to the output dtype) even when they exceed $2^{24}$.

---

### Constraints

- ✅ `torch.round`, `torch.clamp`, `amax`, and integer matmuls (`int32 @ int32` works on CPU in
  recent PyTorch; int64 does too).
- ✅ Use `torch.quantize_per_tensor` / `torch.quantize_per_channel` only to check yourself.
- ❌ No `torch.ao.quantization`, `torch.quantize_per_*` or `torch._int_mm` inside your functions.
- ❌ Don't dequantize inside `int8_linear` — the matmul itself must be integer.
- ❌ Don't accumulate in float32: products reach $255 \cdot 127$, and sums over $K$ pass $2^{24}$,
  beyond which float32 cannot represent every integer.
- ❌ Don't matmul `int8 @ int8` or subtract a zero point from a `uint8` tensor directly —
  PyTorch keeps the 8-bit dtype and silently wraps around.

---

<details>
  <summary>💡 Hint</summary>

  - Casting with `.to(torch.int8)` truncates toward zero; round first.
  - For per-channel scales, think of `amax` with a *list* of dims, then reshape the `(C,)` scale
    so it broadcasts against `x` (size `C` at `axis`, 1 everywhere else).
  - Why must the asymmetric zero point be rounded to an integer *before* you quantize? What
    would happen to zero-padding or ReLU outputs otherwise?
  - What dtype does `uint8_tensor - 7` have? And `int8_tensor @ int8_tensor`? Try it.

</details>

In [ ]:
import warnings

import torch

In [ ]:
torch.manual_seed(0)

# A Linear layer's weight with one outlier output channel (a common sight in LLMs)
W = torch.randn(64, 128) * 0.02
W[7] *= 50.0

# Activations: a batch of post-ReLU features (non-negative) and a signed input
acts_relu = torch.relu(torch.randn(16, 128))
acts_signed = torch.randn(16, 128) * 1.5 + 0.3

print(f"W: {tuple(W.shape)}, max|W| = {W.abs().max():.3f}, typical |W| = {W.abs().median():.4f}")
print(f"ReLU activations in [{acts_relu.min():.2f}, {acts_relu.max():.2f}]")

In [ ]:
def quantize_symmetric(x, num_bits=8, axis=None):
    """Symmetric (zero point 0) quantization, per tensor or per channel.

    Args:
        x: float tensor of any shape.
        num_bits: code width b; codes live in [-(2^(b-1) - 1), 2^(b-1) - 1].
        axis: None for one scale, or the dimension that keeps its own scale (may be negative).

    Returns:
        q: torch.int8 tensor with x's shape.
        scale: 0-dim tensor (axis=None) or 1-D tensor of shape (x.shape[axis],).
    """
    # TODO: qmax = 2^(b-1) - 1
    # TODO: amax = max|x| over everything (axis=None) or over all dims except `axis`
    # TODO: scale = amax / qmax, with all-zero tensors/channels guarded (scale 1.0)
    # TODO: q = clamp(round(x / scale), -qmax, qmax) -> int8 (reshape scale to broadcast)
    raise NotImplementedError

In [ ]:
def quantize_asymmetric(x, num_bits=8):
    """Asymmetric (affine) per-tensor quantization to unsigned codes.

    Args:
        x: float tensor of any shape.
        num_bits: code width b; codes live in [0, 2^b - 1].

    Returns:
        q: torch.uint8 tensor with x's shape.
        scale: 0-dim float tensor.
        zero_point: int — the code that represents 0.0 exactly.
    """
    # TODO: widen [x.min(), x.max()] to include 0, scale = range / (2^b - 1) (guard range == 0)
    # TODO: integer zero_point = clamp(round(-x_min / scale), 0, 2^b - 1)
    # TODO: q = clamp(round(x / scale) + zero_point, 0, 2^b - 1) -> uint8
    raise NotImplementedError

In [ ]:
def dequantize(q, scale, zero_point=None, axis=None):
    """Map integer codes back to float32: (q - zero_point) * scale.

    Args:
        q: integer tensor (int8 or uint8).
        scale: number / 0-dim tensor, or a 1-D tensor of per-channel scales along `axis`.
        zero_point: integer or None (= 0).
        axis: dimension the per-channel scale runs along, or None.

    Returns:
        float32 tensor with q's shape.
    """
    # TODO: upcast q before subtracting the zero point; broadcast a per-channel scale along axis
    raise NotImplementedError

In [ ]:
def int8_linear(x_q, x_scale, x_zero_point, w_q, w_scale):
    """y ≈ dequant(x) @ dequant(w).T with exact integer accumulation and one float rescale.

    Args:
        x_q: (M, K) uint8 activation codes (per-tensor scale x_scale, zero point x_zero_point).
        x_scale: float or 0-dim tensor.
        x_zero_point: int, 0-dim integer tensor, or None (= 0).
        w_q: (N, K) int8 symmetric weight codes (nn.Linear layout).
        w_scale: float / 0-dim tensor, or (N,) per-output-channel scales.

    Returns:
        (M, N) float32 tensor.
    """
    # TODO: upcast both operands to a wide integer type, subtract the activation zero point
    # TODO: integer matmul -> exact (M, N) accumulator
    # TODO: one rescale by x_scale * w_scale (per-channel w_scale broadcasts over the columns)
    raise NotImplementedError

In [ ]:
# Test 1: symmetric per-tensor — dtype, range, error bound, agreement with PyTorch
x = torch.randn(1000) * 3
q, s = quantize_symmetric(x)
assert q.dtype == torch.int8 and q.shape == x.shape
assert q.abs().max() == 127, "the largest |x| should map to ±127"
assert torch.allclose(s, x.abs().max() / 127)
err = (dequantize(q, s) - x).abs()
assert (err <= s / 2 + 1e-6).all(), f"rounding error must be <= scale/2, got {err.max() / s:.3f}·scale"
q4, s4 = quantize_symmetric(x, num_bits=4)
assert q4.abs().max() == 7 and ((dequantize(q4, s4) - x).abs() <= s4 / 2 + 1e-6).all()
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ref = torch.quantize_per_tensor(x, float(s), 0, torch.qint8).int_repr()
    assert (ref.int() - q.int()).abs().max() <= 1 and (ref != q).float().mean() < 1e-3
    print("matches torch.quantize_per_tensor")
except (AttributeError, RuntimeError, NotImplementedError):
    print("(torch.quantize_per_tensor unavailable here — skipped the comparison)")
print(f"Test 1 passed: scale = {float(s):.4f}, max error = {err.max():.4f} <= scale/2")

In [ ]:
# Test 2: per-channel vs per-tensor on the weight with an outlier channel
q_t, s_t = quantize_symmetric(W)                 # one scale
q_c, s_c = quantize_symmetric(W, axis=0)         # one scale per output channel
assert s_c.shape == (W.shape[0],)
assert torch.allclose(s_c, W.abs().amax(dim=1) / 127)
mse_t = (dequantize(q_t, s_t) - W).pow(2).mean(dim=1)
mse_c = (dequantize(q_c, s_c, axis=0) - W).pow(2).mean(dim=1)
assert (mse_c <= mse_t + 1e-12).all(), "per-channel must never be worse than per-tensor"
normal = torch.arange(W.shape[0]) != 7
print(f"MSE on the 63 normal channels: per-tensor {mse_t[normal].mean():.2e}, "
      f"per-channel {mse_c[normal].mean():.2e} ({mse_t[normal].mean() / mse_c[normal].mean():.0f}x better)")
print(f"fraction of normal weights rounded to 0 per-tensor: {(q_t[normal] == 0).float().mean():.0%}")
assert mse_t[normal].mean() > 100 * mse_c[normal].mean()
q_ax1, s_ax1 = quantize_symmetric(W, axis=-1)    # per input column, negative axis
assert s_ax1.shape == (W.shape[1],)
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ref = torch.quantize_per_channel(W, s_c.double(), torch.zeros(64, dtype=torch.long),
                                         0, torch.qint8).int_repr()
    assert (ref.int() - q_c.int()).abs().max() <= 1
    print("matches torch.quantize_per_channel")
except (AttributeError, RuntimeError, NotImplementedError):
    print("(torch.quantize_per_channel unavailable here — skipped the comparison)")
z = torch.zeros(4, 6); z[1] = 1.0
qz, sz = quantize_symmetric(z, axis=0)
assert torch.isfinite(sz).all() and (sz > 0).all(), "all-zero channels need a positive finite scale"
print("Test 2 passed")

In [ ]:
# Test 3: asymmetric — uint8 codes, integer zero point, 0.0 round-trips exactly
xa = acts_signed.clone()
xa[:, ::5] = 0.0                                 # exact zeros (padding / ReLU)
qa, sa, za = quantize_asymmetric(xa)
assert qa.dtype == torch.uint8 and float(za) == int(za) and 0 <= int(za) <= 255
x_lo, x_hi = min(xa.min(), 0), max(xa.max(), 0)
assert torch.allclose(sa, (x_hi - x_lo) / 255)
deq = dequantize(qa, sa, za)
assert (deq[:, ::5] == 0).all(), "0.0 must dequantize to exactly 0.0"
assert ((deq - xa).abs() <= sa / 2 + 1e-6).all()
_, s_pos, z_pos = quantize_asymmetric(torch.rand(100) + 2.0)    # all positive: range is [0, max]
assert int(z_pos) == 0, f"all-positive input -> zero point 0, got {z_pos}"
# On non-negative activations asymmetric uses all 256 codes; symmetric wastes the negative half.
q_s, s_s = quantize_symmetric(acts_relu)
q_a, s_a, z_a = quantize_asymmetric(acts_relu)
mse_s = (dequantize(q_s, s_s) - acts_relu).pow(2).mean()
mse_a = (dequantize(q_a, s_a, z_a) - acts_relu).pow(2).mean()
print(f"post-ReLU activations: symmetric MSE {mse_s:.2e}, asymmetric MSE {mse_a:.2e}")
assert mse_a < 0.5 * mse_s
q0, s0, z0 = quantize_asymmetric(torch.zeros(3, 3))
assert torch.isfinite(dequantize(q0, s0, z0)).all()
print("Test 3 passed")

In [ ]:
# Test 4: int8 linear layer == dequantized float matmul, with an exact integer accumulator
x_q, x_s, x_z = quantize_asymmetric(acts_signed)       # (16, 128) uint8
w_q, w_s = quantize_symmetric(W, axis=0)               # (64, 128) int8, (64,) scales
y = int8_linear(x_q, x_s, x_z, w_q, w_s)
assert y.shape == (16, 64) and y.dtype == torch.float32
y_ref = dequantize(x_q, x_s, x_z).double() @ dequantize(w_q, w_s, axis=0).double().T
assert torch.allclose(y.double(), y_ref, rtol=1e-5, atol=1e-5 * y_ref.abs().max()), \
    "int8_linear must equal dequant(x) @ dequant(w).T up to float rounding"
y_fp = acts_signed @ W.T
print(f"relative error of the int8 layer vs the fp32 layer: {(y - y_fp).norm() / y_fp.norm():.2%}")

# Exactness: unit scales expose the accumulator. These dot products exceed 2^24.
g = torch.Generator().manual_seed(1)
xq_big = torch.randint(200, 256, (8, 4096), generator=g).to(torch.uint8)
wq_big = torch.randint(90, 128, (16, 4096), generator=g).to(torch.int8)
exact = (xq_big.long() - 3) @ wq_big.long().T
y_big = int8_linear(xq_big, 1.0, 3, wq_big, torch.ones(16))
assert torch.equal(y_big, exact.to(y_big.dtype)), "the integer accumulation is not exact"
fp32_acc = (xq_big.float() - 3) @ wq_big.float().T
print(f"float32 accumulation would be wrong in {(fp32_acc != exact.float()).sum()} of {exact.numel()} outputs")
print("Test 4 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("int8-quantization", quantize_symmetric=quantize_symmetric,
      quantize_asymmetric=quantize_asymmetric, dequantize=dequantize, int8_linear=int8_linear)